# Real-hazard corpus-reliance — external validity (Exp 4)

Runs **real, publicly-documented charted dangers** through the necessity instrument, on a real
Bedrock model, so the corpus-reliance claim isn't only on a synthetic hazard. Two phases, one model,
no GPU:

1. **Closed-book screen** — ask the model, with *no corpus*, whether it already knows the danger at
   each location. If it names the danger (`danger_terms`) it's **leaked** (parametric) → **DROP**; if
   not it's a **USABLE** corpus-bound target. Screening against the *same* model that runs the
   instrument is the right check (a danger one model knows another may not). Which candidates drop is
   itself reportable data.
2. **Instrument run** — each USABLE hazard goes through `HAZARDS_FILE=<line> PROBES=hazard npm run
   colreg:leakage`: the danger sits on the track, scored by the barrier but shown **only** in the
   corpus. necessity = regret(ablated) − regret(present).

Candidates + citations live in `experiments/unlearning/real_hazards.jsonl` /
`real_hazards.SOURCES.md` (Elwha / Fullastern / Whittle real; Seven Stones = a famous **control**,
expected to DROP). Auth = standard AWS chain (nothing pasted).


## 1 · Config + clone  (set `BEDROCK_MODEL`, then Run All)


In [ ]:
import os, re, json, subprocess, datetime

REPO_URL   = 'https://github.com/wrgr/socratic-scenarios.git'
BRANCH     = 'main'
AWS_REGION = os.environ.get('AWS_REGION', 'us-east-1')

# The single model to screen AND run the instrument against (screen and instrument must agree on
# 'does THIS model know it'). Any Bedrock inference-profile id — see the model-scan notebook for the
# verified class x size matrix. Default: a large model (most likely to already know a real danger,
# so the screen has teeth).
BEDROCK_MODEL = 'us.anthropic.claude-opus-4-5-20251101-v1:0'

# Candidate hazards: repo file by default (Elwha, Fullastern, Whittle real + Seven Stones control).
# Each line: {"location", "disclosure", "danger_terms"}. Add your own from official charts / NtM.
HAZARDS_REL = 'experiments/unlearning/real_hazards.jsonl'

if not os.path.isdir('socratic-scenarios'):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL], check=True)
subprocess.run(['git','fetch','--depth','1','origin',BRANCH], cwd='socratic-scenarios', check=True)
subprocess.run(['git','reset','--hard','FETCH_HEAD'], cwd='socratic-scenarios', check=True)
REPO = os.path.abspath('socratic-scenarios')
HAZARDS_FILE = os.path.join(REPO, HAZARDS_REL)
cands = [json.loads(l) for l in open(HAZARDS_FILE) if l.strip()]
print('repo:', REPO, '| region:', AWS_REGION, '| model:', BEDROCK_MODEL)
print('candidates:', len(cands), '->', ', '.join(c['location'][:28] for c in cands))


## 2 · Deps + AWS credentials
`npm install` for the instrument; `boto3` for the closed-book screen. Credentials come from the
standard AWS chain (env / `~/.aws` / instance role) — nothing is pasted or stored.


In [ ]:
subprocess.run(['npm','install','--no-audit','--no-fund','--loglevel=error'], cwd=REPO, check=True)
try:
    import boto3
except ImportError:
    subprocess.run(['pip','install','-q','boto3'], check=True); import boto3

brt = boto3.client('bedrock-runtime', region_name=AWS_REGION)
who = subprocess.run(['aws','sts','get-caller-identity'], capture_output=True, text=True)
if who.returncode == 0:
    print('AWS identity OK:', json.loads(who.stdout).get('Arn','?'))
else:
    have = [k for k in ('AWS_ACCESS_KEY_ID','AWS_PROFILE','AWS_ROLE_ARN','AWS_WEB_IDENTITY_TOKEN_FILE') if os.environ.get(k)]
    print('aws cli check unavailable; env-chain markers present:', have or 'NONE — set creds first')


## 3 · Provenance (saved with the results)


In [ ]:
run_utc = datetime.datetime.now(datetime.timezone.utc)
commit  = subprocess.run(['git','rev-parse','HEAD'], cwd=REPO, capture_output=True, text=True).stdout.strip()
PROV = {
    'run_utc':   run_utc.isoformat(),
    'run_local': datetime.datetime.now().astimezone().isoformat(),
    'git_commit': commit, 'branch': BRANCH, 'aws_region': AWS_REGION,
    'model': BEDROCK_MODEL, 'hazards_file': HAZARDS_REL, 'n_candidates': len(cands),
    'instrument': 'colreg:leakage PROBES=hazard (necessity=regret-delta; redundant/unusable=regret-with)',
    'screen': 'closed-book Bedrock converse vs danger_terms (screen_hazards.knows_hazard)',
}
print(json.dumps(PROV, indent=2))


## 4 · Phase 1 — closed-book screen
Each call is logged with a UTC stamp for traceability. Reuses `knows_hazard` / `CLOSED_BOOK_Q` from
the repo's `screen_hazards.py` so the notebook and the CLI screen can't drift.


In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO, 'experiments', 'unlearning'))
from screen_hazards import knows_hazard, CLOSED_BOOK_Q   # same classifier as the CLI screen

cmd_log = []   # traceability: exact call + UTC for every model touch (screen + run)

def bedrock_ask(prompt, max_tokens=220):
    r = brt.converse(modelId=BEDROCK_MODEL,
                     messages=[{'role':'user','content':[{'text':prompt}]}],
                     inferenceConfig={'maxTokens':max_tokens,'temperature':0.0})
    return r['output']['message']['content'][0]['text']

screen = []   # one row per candidate
for c in cands:
    ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
    call = f"bedrock:converse model={BEDROCK_MODEL} region={AWS_REGION} closed-book('{c['location'][:32]}...')"
    print(f"SCREEN {ts}  {call}", flush=True)
    cmd_log.append({'utc': ts, 'phase': 'screen', 'location': c['location'], 'call': call})
    ans = bedrock_ask(CLOSED_BOOK_Q.format(location=c['location']))
    leaked = knows_hazard(ans, c.get('danger_terms'))
    screen.append({'location': c['location'], 'leaked': leaked, 'closed_book': ans.replace(chr(10),' ')})
    print(f"  -> [{'DROP (leaked)' if leaked else 'USABLE       '}] {ans[:150].replace(chr(10),' ')}\n")

usable = [c for c, s in zip(cands, screen) if not s['leaked']]
print(f"{len(usable)}/{len(cands)} USABLE (model did not name the danger closed-book); "
      f"{len(cands)-len(usable)} dropped as already-known.")


## 5 · Phase 2 — run each USABLE hazard through the instrument
`HAZARDS_FILE=<one-line json> PROBES=hazard npm run --silent colreg:leakage`, one per usable hazard.
The exact command is printed + logged before each call.


In [ ]:
DASH = r'[-−]'   # ASCII hyphen or unicode minus

def parse_hazard(out):
    r = {}
    m = re.search(r'regret-delta\s+[\d.]+\s*\(without\)\s*'+DASH+r'\s*([\d.]+)\s*\(with\)\s*=\s*(-?[\d.]+)', out)
    if m: r['necessity'] = float(m.group(2)); r['regret_with'] = float(m.group(1))
    v = re.search(r'VERDICT:\s*([A-Z-]+)(?:\s*\((\w+):\s*regret-with\s*([\d.]+)\))?', out)
    if v:
        r['verdict'] = v.group(1); r['leak_mode'] = v.group(2) or ''
        if v.group(3): r['regret_with'] = float(v.group(3))
    return r

outdir = os.path.join(REPO, 'results', 'real-hazards'); os.makedirs(outdir, exist_ok=True)

def slug(s):
    return re.sub(r'[^a-z0-9]+', '-', s.lower()).strip('-')[:40]

rows = []
for i, c in enumerate(usable):
    line = json.dumps({'location': c['location'], 'disclosure': c['disclosure']})
    hz_path = os.path.join(outdir, f'hz_{i}.json'); open(hz_path, 'w').write(line)
    ts  = datetime.datetime.now(datetime.timezone.utc).isoformat()
    cmd = f"HAZARDS_FILE={os.path.relpath(hz_path, REPO)} AWS_REGION={AWS_REGION} BEDROCK_MODEL={BEDROCK_MODEL} PROBES=hazard npm run --silent colreg:leakage"
    print(f"RUN {ts}  [{slug(c['location']):40}]\n    {cmd}", flush=True)
    cmd_log.append({'utc': ts, 'phase': 'run', 'location': c['location'], 'cmd': cmd})
    env = dict(os.environ, HAZARDS_FILE=hz_path, AWS_REGION=AWS_REGION, BEDROCK_MODEL=BEDROCK_MODEL, PROBES='hazard')
    p = subprocess.run(['npm','run','--silent','colreg:leakage'], cwd=REPO, env=env, capture_output=True, text=True)
    out = (p.stdout or '') + '\n' + (p.stderr or '')
    open(os.path.join(outdir, f'{slug(c["location"])}__hazard.txt'), 'w').write(out)
    row = {'location': c['location']}
    if 'VERDICT' not in out and p.returncode != 0:
        row['error'] = (p.stderr or p.stdout or 'failed')[-200:]
    else:
        row.update(parse_hazard(out))
    rows.append(row)
    print(f"  -> necessity={row.get('necessity','?')} verdict={row.get('verdict','?')}/{row.get('leak_mode','')} "
          f"regret-with={row.get('regret_with','?')}{'  ERR' if 'error' in row else ''}\n")


## 6 · Summary + paste-back  (provenance + screen + full command log)


In [ ]:
def to_md(rs, cols):
    cols = [c for c in cols if any(c in r for r in rs)]
    head = '| ' + ' | '.join(cols) + ' |'
    sep  = '| ' + ' | '.join('---' for _ in cols) + ' |'
    body = '\n'.join('| ' + ' | '.join(str(r.get(c,'')) for c in cols) + ' |' for r in rs)
    return '\n'.join([head, sep, body])

stamp = run_utc.strftime('%Y%m%dT%H%M%SZ')
screen_rows = [{'location': s['location'], 'screen': 'DROP (leaked)' if s['leaked'] else 'USABLE'} for s in screen]
payload = {'provenance': PROV, 'command_log': cmd_log, 'screen': screen, 'rows': rows}
json.dump(payload, open(os.path.join(outdir, f'realhazards_{stamp}.json'), 'w'), indent=2)

print('==================== PASTE THIS BACK ====================')
print(f"real-hazard external-validity run | run_utc={PROV['run_utc']} | local={PROV['run_local']}")
print(f"commit={PROV['git_commit'][:9]} branch={BRANCH} region={AWS_REGION} model={BEDROCK_MODEL}")
print(f"screen: {sum(not s['leaked'] for s in screen)}/{len(screen)} usable | {len(cmd_log)} model calls logged\n")
print('SCREEN (all candidates):')
print(to_md(screen_rows, ['location','screen']))
print('\nNECESSITY (usable hazards):')
print(to_md(rows, ['location','necessity','verdict','leak_mode','regret_with','error']) if rows
      else '(none usable — every candidate was already known to this model)')
print('\n<details><summary>provenance + screen answers + command log + rows (machine-readable)</summary>\n')
print('```json'); print(json.dumps(payload, indent=2)); print('```\n</details>')
print(f"\nsaved: results/real-hazards/realhazards_{stamp}.json  (+ per-hazard raw .txt)")


## Done
Paste the **PASTE THIS BACK** block into the chat. Expected shape: the obscure real dangers
(Elwha / Fullastern / Whittle) read **corpus-bound** (necessity ≫ 0, the model needed the corpus),
while **Seven Stones** — the famous control — should **DROP** in the screen (the model names it
closed-book). Any candidate that drops is reportable: *N of M screened out as already-known*.
